In [2]:
%pip install torch tiktoken pandas numpy

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
    --------------------------------------- 1.6/124.1 MB 8.4 MB/s eta 0:00:15
   -- ------------------------------------- 6.8/124.1 MB 16.0 MB/s eta 0:00:08
   --- ------------------------------------ 11.5/124.1 MB 18.1 MB/s eta 0:00:07
   ---- ----------------------------------- 15.2/124.1 MB 17.2 MB/s eta 0:00:07
   ----- ---------------------------------- 17.3/124.1 MB 15.8 MB/s eta 0:00:07
   ------ --------------------------------- 20.7/124.1 MB 15.7 MB/s eta 0:00:07
   -------- ------------------------------- 24.9/124.1 MB 16.4 MB/s eta 0:00:07
   --------- ------------------------------ 28.8/124.1 MB 16.5 MB/s eta 0:00:06
   ---------- ----------------------------- 32.5/124.1 MB 16.6 MB/s eta 0:00:06
   ----------- ---------------------------- 35.7/124.1 MB 16.5 MB/s eta 0:00:06
   ------------ --------------------------- 39.1/124.1 MB 16.4 MB/s e


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import tiktoken

print("PyTorch version:", torch.__version__)
print("Pandas version:", pd.__version__)
print("tiktoken imported successfully!")

PyTorch version: 2.14.1+cpu
Pandas version: 3.0.3
tiktoken imported successfully!


In [4]:
# Load the IMDB dataset
df = pd.read_csv("IMDB Dataset.csv")

# Display basic information
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# Display first 5 rows
df.head()

Dataset shape: (50000, 2)

Columns:
['review', 'sentiment']


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
# Subsample 2,500 reviews for faster training
df = df.sample(n=2500, random_state=42).reset_index(drop=True)

print("Sampled dataset shape:", df.shape)
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Sampled dataset shape: (2500, 2)

Sentiment distribution:
sentiment
positive    1261
negative    1239
Name: count, dtype: int64


In [6]:
# Convert sentiment labels to numeric values
# positive = 1
# negative = 0

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df[["sentiment", "label"]].head())
print("\nLabel distribution:")
print(df["label"].value_counts())

  sentiment  label
0  positive      1
1  positive      1
2  negative      0
3  positive      1
4  negative      0

Label distribution:
label
1    1261
0    1239
Name: count, dtype: int64


In [8]:
# Task 3: Split the data into training and test sets

# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Calculate the split point (80% training, 20% testing)
split_index = int(0.8 * len(df))

# Create training and testing datasets
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# Extract reviews and labels
X_train = train_df["review"].tolist()
y_train = train_df["label"].tolist()

X_test = test_df["review"].tolist()
y_test = test_df["label"].tolist()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining labels:")
print(pd.Series(y_train).value_counts())

print("\nTesting labels:")
print(pd.Series(y_test).value_counts())

Training samples: 2000
Testing samples: 500

Training labels:
1    1014
0     986
Name: count, dtype: int64

Testing labels:
0    253
1    247
Name: count, dtype: int64


In [9]:
# Task 4: Initialize tiktoken tokenizer
enc = tiktoken.get_encoding("cl100k_base")

print("Tokenizer:", enc.name)

# Test the tokenizer on one review
sample_tokens = enc.encode(X_train[0])

print("Number of tokens in sample review:", len(sample_tokens))
print("First 20 token IDs:", sample_tokens[:20])

Tokenizer: cl100k_base
Number of tokens in sample review: 923
First 20 token IDs: [3947, 1288, 387, 264, 6037, 430, 5415, 5115, 9539, 430, 9698, 1093, 60776, 34819, 527, 10171, 311, 387, 1903, 304]


In [10]:
# Task 4 & 5: Tokenize reviews and make all sequences the same length

MAX_LEN = 200

def tokenize_and_pad(text):
    # Convert text to token IDs
    tokens = enc.encode(text)

    # Truncate if longer than MAX_LEN
    tokens = tokens[:MAX_LEN]

    # Pad with 0 if shorter than MAX_LEN
    if len(tokens) < MAX_LEN:
        tokens = tokens + [0] * (MAX_LEN - len(tokens))

    return tokens


# Tokenize training and testing reviews
X_train_tokens = [tokenize_and_pad(review) for review in X_train]
X_test_tokens = [tokenize_and_pad(review) for review in X_test]

print("Number of training sequences:", len(X_train_tokens))
print("Number of testing sequences:", len(X_test_tokens))
print("Length of each sequence:", len(X_train_tokens[0]))

print("\nFirst 20 tokens of first review:")
print(X_train_tokens[0][:20])

Number of training sequences: 2000
Number of testing sequences: 500
Length of each sequence: 200

First 20 tokens of first review:
[3947, 1288, 387, 264, 6037, 430, 5415, 5115, 9539, 430, 9698, 1093, 60776, 34819, 527, 10171, 311, 387, 1903, 304]


In [11]:
# Task 6: Convert token sequences and labels into PyTorch tensors

X_train_tensor = torch.tensor(X_train_tokens, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_tokens, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("Training input shape:", X_train_tensor.shape)
print("Testing input shape:", X_test_tensor.shape)

print("Training labels shape:", y_train_tensor.shape)
print("Testing labels shape:", y_test_tensor.shape)

print("\nData types:")
print("X_train:", X_train_tensor.dtype)
print("y_train:", y_train_tensor.dtype)

Training input shape: torch.Size([2000, 200])
Testing input shape: torch.Size([500, 200])
Training labels shape: torch.Size([2000])
Testing labels shape: torch.Size([500])

Data types:
X_train: torch.int64
y_train: torch.int64


In [12]:
# Task 7: Build the RNN-based text classifier

class RNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()

        # Convert token IDs into dense vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # RNN layer
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Linear output layer
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, sequence_length)

        embedded = self.embedding(x)

        # embedded shape: (batch_size, sequence_length, embedding_dim)
        output, hidden = self.rnn(embedded)

        # Take the final RNN output
        last_output = output[:, -1, :]

        # Classification
        logits = self.fc(last_output)

        return logits


# Get the vocabulary size from the tokenizer
vocab_size = enc.n_vocab

embedding_dim = 64
hidden_dim = 64
num_classes = 2

model = RNNTextClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes
)

print(model)

RNNTextClassifier(
  (embedding): Embedding(100277, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


In [13]:
# Task 8: Set up loss function, optimizer, and DataLoader

from torch.utils.data import TensorDataset, DataLoader

# Create a dataset from tensors
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# Create batches
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Batch size:", 32)
print("Number of batches:", len(train_loader))
print("Loss function:", criterion)
print("Optimizer:", optimizer)

Batch size: 32
Number of batches: 63
Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [14]:
# Task 8: Train the RNN model

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        # Clear previous gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch_X)
        
        # Calculate loss
        loss = criterion(outputs, batch_y)
        
        # Backpropagation
        loss.backward()
        
        # Update model parameters
        optimizer.step()
        
        total_loss += loss.item()
    
    average_loss = total_loss / len(train_loader)
    
    print(
        f"Epoch [{epoch + 1}/{num_epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/5], Loss: 0.7040
Epoch [2/5], Loss: 0.6704
Epoch [3/5], Loss: 0.6452
Epoch [4/5], Loss: 0.6080
Epoch [5/5], Loss: 0.5698


In [15]:
# Task 9: Evaluate the model on the test set

model.eval()

with torch.no_grad():
    # Get predictions for all test reviews
    test_outputs = model(X_test_tensor)

    # Get the class with the highest score
    predictions = torch.argmax(test_outputs, dim=1)

    # Calculate accuracy
    correct = (predictions == y_test_tensor).sum().item()
    total = y_test_tensor.size(0)

    accuracy = correct / total * 100

print(f"Correct predictions: {correct}/{total}")
print(f"Test Accuracy: {accuracy:.2f}%")

Correct predictions: 268/500
Test Accuracy: 53.60%


In [16]:
# Task 10: Test the model on new reviews

new_reviews = [
    "This movie was absolutely fantastic! The story was amazing and the actors were excellent.",
    "I really disliked this movie. It was boring, slow, and a complete waste of time.",
    "The movie was okay, but the story was not very interesting."
]

# Tokenize and pad the new reviews
new_reviews_tokens = [
    tokenize_and_pad(review) for review in new_reviews
]

# Convert to PyTorch tensor
new_reviews_tensor = torch.tensor(
    new_reviews_tokens,
    dtype=torch.long
)

# Make predictions
model.eval()

with torch.no_grad():
    outputs = model(new_reviews_tensor)
    predictions = torch.argmax(outputs, dim=1)

# Display predictions
sentiment_labels = {
    0: "Negative",
    1: "Positive"
}

for review, prediction in zip(new_reviews, predictions):
    print("Review:", review)
    print("Predicted sentiment:", sentiment_labels[prediction.item()])
    print("-" * 80)

Review: This movie was absolutely fantastic! The story was amazing and the actors were excellent.
Predicted sentiment: Negative
--------------------------------------------------------------------------------
Review: I really disliked this movie. It was boring, slow, and a complete waste of time.
Predicted sentiment: Negative
--------------------------------------------------------------------------------
Review: The movie was okay, but the story was not very interesting.
Predicted sentiment: Negative
--------------------------------------------------------------------------------
